# 05 · Distil the Extractor into a small local model with QLoRA (component B)

**Purpose.** Build a training set from the API Extractor's output on training-split papers, keeping only extractions where EVERY claim passed the Verifier -- the Verifier is the data filter, so the student never learns to hallucinate evidence the teacher got away with. Then QLoRA fine-tune a small instruct model and compare it with the API extractor on held-out papers.

| | |
|---|---|
| **Inputs** | `data/extractions_train_api.jsonl` and `data/claim_verdicts_train_api.json` (notebook 03) |
| **Outputs** | `data/distill_train.jsonl`, `models/distilled_extractor/`, `results/stage6_distill.json` |
| **Expected runtime (Colab T4)** | data build ~5 min; QLoRA 1.5-3 h for ~400 examples on a T4; comparison ~20 min |
| **Required secrets** | `GROQ_API_KEY` (for the API side of the comparison), `HF_TOKEN` if the base model is gated |
| **GPU** | **T4 required.** 4-bit QLoRA needs bitsandbytes, which is CUDA-only. On CPU/MPS the trainer falls back to fp32 LoRA and reports `quantised_4bit: false` rather than pretending it quantised. |

Everything is cached under `/content/drive/MyDrive/litreview/` and every stage is
**resumable**: if Colab disconnects, re-run all cells and it continues where it
stopped. Only the cell marked CONFIG needs editing.


In [ ]:
# --- Colab setup: Drive, project code, secrets. Safe to re-run. ---
import json, os, subprocess, sys, zipfile
from pathlib import Path
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = "/content/drive/MyDrive/litreview" if ON_COLAB else os.path.abspath("..")
os.makedirs(ROOT, exist_ok=True)
os.environ["LITREVIEW_ROOT"] = ROOT
REPO = os.path.join(ROOT, "repo")

# Optional: a git URL to clone the project from. Leave blank to use the folder or
# the zip on Drive instead.
REPO_URL = ""

def _find_project() -> str | None:
    """Locate the project code: an existing folder, a zip on Drive, or a clone."""
    if os.path.exists(os.path.join(REPO, "pyproject.toml")):
        return REPO
    # a litreview_*.zip uploaded anywhere under MyDrive/litreview
    zips = sorted(Path(ROOT).glob("litreview*.zip")) + sorted(Path(ROOT).glob("**/litreview*.zip"))
    for z in zips:
        print(f"extracting {z.name} ...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(ROOT)
        # the zip may contain a top-level folder; find the pyproject inside
        for cand in [REPO, ROOT] + [str(p.parent) for p in Path(ROOT).glob("*/pyproject.toml")]:
            if os.path.exists(os.path.join(cand, "pyproject.toml")):
                return cand
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=False)
        if os.path.exists(os.path.join(REPO, "pyproject.toml")):
            return REPO
    return None

if ON_COLAB:
    found = _find_project()
    if not found:
        raise SystemExit(
            "Could not find the project code. Do one of these, then re-run this cell:
"
            "  1. upload litreview_colab.zip into MyDrive/litreview/ (easiest), or
"
            "  2. copy the project folder to MyDrive/litreview/repo, or
"
            "  3. set REPO_URL above to a git URL.
"
            "The notebooks deliberately contain no second copy of the code.")
    REPO = found
    print("project code:", REPO)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", f"{REPO}[ml,eval]"],
                   check=True)

import litreview
from litreview.config import SETTINGS, paths, secret
P = paths().mkdirs()
print("data root:", P.root)
print("already present:", {f.name: f.stat().st_size // 1024 for f in sorted(P.data.glob("*.jsonl"))})


In [ ]:
# ======================= CONFIG (the only cell to edit) =======================
PROVIDER   = "groq"
MODEL      = ""
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # check availability; 3B also fits a T4 at max_len 2048
SPLIT      = "train"
EPOCHS, LR, MAX_LEN = 2, 2e-4, 3072
LORA_R, LORA_ALPHA  = 16, 32
GRAD_ACCUM = 8
ALLOW_PARTIAL = False        # True keeps "partially supported" claims too (more data, looser)
N_COMPARE  = 20              # held-out papers for the head-to-head
SEED = 0
# ==============================================================================


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "bitsandbytes", "trl"], check=False)   # CUDA-only; harmless elsewhere
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
# Provider comes from one config. Keys are read from Colab secrets, never hard-coded.
import os
os.environ["LITREVIEW_PROVIDER"] = PROVIDER
if MODEL:
    os.environ["LITREVIEW_MODEL"] = MODEL

from litreview.config import Settings
from litreview.llm import install_call_logger, probe, usage_summary
import litreview.config as _cfg
_cfg.SETTINGS = Settings()                      # re-read after setting the env vars
import litreview.llm as _llm
_llm.SETTINGS = _cfg.SETTINGS

install_call_logger(P.results / "llm_calls.jsonl")
print(json.dumps(_cfg.SETTINGS.describe(), indent=1))
print(json.dumps(probe(_cfg.SETTINGS), indent=1))


### Build the Verifier-filtered training set

Only papers whose every claim came back `supported` survive. The report records
how many were dropped and why, because that number *is* the stage's argument.

In [ ]:
from litreview.distill import build_dataset
from litreview.extract import load_records

records = load_records(P.data / f"extractions_{SPLIT}_api.jsonl")
vpath = P.data / f"claim_verdicts_{SPLIT}_api.json"
verdicts = json.loads(vpath.read_text())["per_paper"] if vpath.exists() else {}
if not verdicts:
    raise SystemExit("no claim verdicts found; run notebook 03's verify cell first")

stats = build_dataset(records, verdicts, P.data / "distill_train.jsonl",
                      require_all_supported=True, allow_partial=ALLOW_PARTIAL, seed=SEED)
print(json.dumps(stats, indent=2))


In [ ]:
rows = [json.loads(l) for l in (P.data / "distill_train.jsonl").open()]
print(f"{len(rows)} training examples")
print("--- one example's target ---")
print(rows[0]["messages"][2]["content"][:900])


### QLoRA fine-tune

4-bit NF4 base with LoRA adapters on the attention and MLP projections, gradient
checkpointing, paged AdamW. Save the adapter to Drive so the evaluation notebook
can load it.

In [ ]:
from litreview.distill import train_qlora
report = train_qlora(P.data / "distill_train.jsonl", P.models / "distilled_extractor",
                     base_model=BASE_MODEL, epochs=EPOCHS, lr=LR, max_len=MAX_LEN,
                     lora_r=LORA_R, lora_alpha=LORA_ALPHA, grad_accum=GRAD_ACCUM, seed=SEED)
print(json.dumps(report, indent=2))


### Head to head on held-out papers

JSON validity rate, claim faithfulness through the same Verifier, field-level
agreement, latency and cost per paper. The distilled model runs locally, so its
monetary cost is zero and its latency is hardware-bound; the API column's cost
comes from the central usage log.

In [ ]:
import sys
from litreview.distill import compare_extractors
from litreview.extract import ApiExtractor, DistilledExtractor
from litreview.verify import Verifier
SCRIPTS = (Path(REPO) / "scripts") if ON_COLAB else Path("../scripts")
sys.path.insert(0, str(SCRIPTS))
from run_stage3 import papers_from_mining

held_out = papers_from_mining(N_COMPARE, "test", SEED)      # test split: never trained on
print(f"{len(held_out)} held-out papers")
cmp = compare_extractors(held_out, ApiExtractor(),
                         DistilledExtractor(P.models / "distilled_extractor"),
                         verifier=Verifier(), out_path=P.results / "stage6_distill.json")
cmp["training"] = report
cmp["dataset"] = stats
(P.results / "stage6_distill.json").write_text(json.dumps(cmp, indent=2))
print(json.dumps({k: v for k, v in cmp.items() if k != "llm_usage"}, indent=2))


### Next

**06_evaluation.ipynb** — pass `--distilled models/distilled_extractor` so system
S4 is included in the comparison.